In [73]:
# BPE Tokenizer — Interview Practice Notebook

# SECTION 1: Load and Inspect Data
# Tasks:
# - Load corpus
# - Print length
# - Unique chars

# YOUR CODE HERE
with open("the-verdict.txt", 'r') as file:
  text = file.read()

print("length: ", len(text))
unique_chars = sorted(set(text))
print(len(unique_chars))

length:  20479
62


In [74]:
# Tasks:
# - Create initial vocabulary of characters
# - Represent words as list of characters + </w>

# YOUR CODE HERE
text = text.split()
text = [list(word) + ["</w>"] for word in text]
text[:5]

[['I', '</w>'],
 ['H', 'A', 'D', '</w>'],
 ['a', 'l', 'w', 'a', 'y', 's', '</w>'],
 ['t', 'h', 'o', 'u', 'g', 'h', 't', '</w>'],
 ['J', 'a', 'c', 'k', '</w>']]

In [75]:
vocab = {}

for li in text:
  vocab[tuple(li)] = vocab.get(tuple(li), 0) + 1
print(vocab)

{('I', '</w>'): 100, ('H', 'A', 'D', '</w>'): 1, ('a', 'l', 'w', 'a', 'y', 's', '</w>'): 6, ('t', 'h', 'o', 'u', 'g', 'h', 't', '</w>'): 7, ('J', 'a', 'c', 'k', '</w>'): 4, ('G', 'i', 's', 'b', 'u', 'r', 'n', '</w>'): 8, ('r', 'a', 't', 'h', 'e', 'r', '</w>'): 5, ('a', '</w>'): 80, ('c', 'h', 'e', 'a', 'p', '</w>'): 1, ('g', 'e', 'n', 'i', 'u', 's', '-', '-', 't', 'h', 'o', 'u', 'g', 'h', '</w>'): 1, ('g', 'o', 'o', 'd', '</w>'): 2, ('f', 'e', 'l', 'l', 'o', 'w', '</w>'): 1, ('e', 'n', 'o', 'u', 'g', 'h', '-', '-', 's', 'o', '</w>'): 1, ('i', 't', '</w>'): 40, ('w', 'a', 's', '</w>'): 64, ('n', 'o', '</w>'): 7, ('g', 'r', 'e', 'a', 't', '</w>'): 2, ('s', 'u', 'r', 'p', 'r', 'i', 's', 'e', '</w>'): 1, ('t', 'o', '</w>'): 97, ('m', 'e', '</w>'): 19, ('h', 'e', 'a', 'r', '</w>'): 4, ('t', 'h', 'a', 't', ',', '</w>'): 7, ('i', 'n', '</w>'): 40, ('t', 'h', 'e', '</w>'): 162, ('h', 'e', 'i', 'g', 'h', 't', '</w>'): 2, ('o', 'f', '</w>'): 95, ('h', 'i', 's', '</w>'): 55, ('g', 'l', 'o', 'r', 

In [76]:
# Tasks:
# - Count frequency of adjacent symbol pairs
# - Return dict: {(a,b): count}

# YOUR CODE HERE
def get_pair_frequencies(corpus):
  pair_freq = {} #{(str,str) -> int}
  for word,freq in corpus.items():
    for pair in zip(word[:], word[1:]):
      pair_freq[pair] = pair_freq.get(pair,0) + freq
  return pair_freq

In [77]:
pair_freq = get_pair_frequencies(vocab)
# print(pair_freq)

In [78]:
# Tasks:
# - Return most frequent pair

# YOUR CODE HERE
def get_most_frequent_pair(pair_freq):
  most_freq_pair = max(pair_freq, key = pair_freq.get)
  return most_freq_pair

most_freq_pair = get_most_frequent_pair(pair_freq)
print(most_freq_pair, pair_freq[most_freq_pair])

('e', '</w>') 614


In [79]:
def merge_pair(pair, vocab):
  new_vocab = {} #{tuple[str] -> freq[int]}
  for word,freq in vocab.items():
    i = 0
    new_word = []
    while i < len(word):
      if i < len(word) - 1 and (word[i] == pair[0] and word[i+1] == pair[1]):
        new_word.append(pair[0] + pair[1])
        i = i + 2
      else:
        new_word.append(word[i])
        i = i + 1
    new_vocab[tuple(new_word)] = new_vocab.get(tuple(new_word),0) + freq
  return new_vocab

In [80]:
# Tasks:
# - Run BPE for N iterations
# - Store merges

# YOUR CODE HERE
def train_bpe(vocab, num_merges):
  merges = list()
  for _ in range(num_merges):
    pair_freq = get_pair_frequencies(vocab)
    if not pair_freq:
      break
    best_pair = get_most_frequent_pair(pair_freq)
    if best_pair is None:
      break

    vocab = merge_pair(best_pair, vocab)
    merges.append(best_pair)

  return vocab,merges

In [81]:
final_vocab, merges = train_bpe(vocab, 20)
type(merges)

list

In [96]:
# Tasks:
# - Tokenize new word using merges

# YOUR CODE HERE
def tokenize(word, merges):

  tokens = list(word) + ["</w>"]

  for pair in merges:
    new_tokens = []
    i = 0
    while i < len(tokens):
      if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1]==pair[1]:
        new_tokens.append(pair[0] + pair[1])
        i += 2
      else:
        new_tokens.append(tokens[i])
        i += 1

    tokens = new_tokens

  return tokens


In [97]:
words = "erereranouereou"
print(tokenize(words,merges))

['er', 'er', 'er', 'an', 'ou', 'er', 'e', 'ou', '</w>']


In [104]:
def decode(tokens):
  return "".join(tokens).replace("</w>", " ").strip()

In [102]:
len(merges)

20

In [103]:
merges

[('e', '</w>'),
 ('d', '</w>'),
 ('t', '</w>'),
 ('t', 'h'),
 ('s', '</w>'),
 ('i', 'n'),
 (',', '</w>'),
 ('o', 'u'),
 ('e', 'r'),
 ('a', 'n'),
 ('.', '</w>'),
 ('o', 'n'),
 ('th', 'e</w>'),
 ('y', '</w>'),
 ('e', 'n'),
 ('e', 'd</w>'),
 ('o', '</w>'),
 ('f', '</w>'),
 ('h', 'a'),
 ('in', 'g')]

In [105]:
tokens = tokenize("lowest", merges)
decoded = decode(tokens)

print(tokens)
print(decoded)

['l', 'o', 'w', 'e', 's', 't</w>']
lowest
